Let's break down the mechanics of the RSA encryption algorithm. 

**The Core Concept of Public Key Cryptography**
Invented by Whitfield Diffie, Martin Hellman, and Ralph Merkle, public key cryptography relies on paired keys that work in a strict one-way relationship:
* **Public Key:** Used to encrypt the plaintext message into ciphertext.
* **Private Key:** Used to decrypt the ciphertext back into readable plaintext.

**The Security Backbone**
RSA derives its security from the extreme difficulty of factoring the product of two massive numbers. The keys are built using a pair of large prime numbers (typically 100 to 200 digits long). 

**Key Generation Step-by-Step**
1. **Choose Primes:** Pick two large, secret prime numbers, **p** and **q**.
2. **Calculate n:** Compute the product **n = p × q**. This number defines the key length.
3. **Choose e:** Select an encryption key exponent, **e**, such that e and (p - 1) × (q - 1) share no common factors. Mathematically:  
   `gcd(e, (p - 1) * (q - 1)) = 1`
4. **Calculate d:** Compute the decryption exponent, **d**, which is the multiplicative inverse of e modulo (p - 1) × (q - 1). This is done using the Extended Euclidean Algorithm.

*Result:* The pair **(e, n)** becomes your shared **Public Key**, while **d** becomes your secret **Private Key**. The original primes p and q must be destroyed or hidden.

**Encryption and Decryption Formulas**
To secure and reveal a message (m), the algorithm utilizes modular exponentiation:
* **Encryption:** `c = (m^e) % n`
* **Decryption:** `m = (c^d) % n`

**Why the Math Works**
Because d is the mathematical inverse of e, raising the ciphertext to the power of d cancels out the encryption exponent, returning the exact original message:
* c^d ≡ (m^e)^d (mod n)
* ≡ m^(e * d) (mod n)
* ≡ m^1 (mod n)
* ≡ m (mod n)

**Preparing Data for RSA**
Before encrypting a text string like "hello world", it must be converted into a single large number (m). 
Instead of standard decimal ASCII (which produces values with unpredictable numbers of digits), RSA implementations convert text into a **hexadecimal format**. Every single character or byte translates reliably into exactly two hexadecimal digits, ensuring a clean and predictable conversion into a number block.


amortized analysis is very useful in analyzing more advanced algorithms.

Other **Good refrences:**
- MIT OpenCourseWare, Mathematics for Computer Science: https://ocw.mit.edu/courses/6-042j-mathematics-for-computer-science-spring-2015/
- MIT OpenCourseWare,Introduction to Computer Science and Programming in Python: https://ocw.mit.edu/courses/6-0001-introduction-to-computer-science-and-programming-in-python-fall-2016/pages/syllabus/
- MIT OpenCourseWare, Introduction to Algorithms: https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-spring-2020/pages/syllabus/
- MIT OpenCourseWare, Design and Analysis of Algorithms: https://ocw.mit.edu/courses/6-046j-design-and-analysis-of-algorithms-spring-2015/pages/syllabus/

**others**
- RSA Algorithm - How does it work? - I'll PROVE it with an Example! -- Cryptography - Practical TLS
Practical Networking: https://youtu.be/Pq8gNbvfaoM?si=h6UatU1z6riXQIvN
- Khan Academy Labs: https://www.youtube.com/@KhanAcademyLabs/search?query=RSA%20Algorithm
- https://www.youtube.com/@khanacademy/search?query=RSA%20Algorithm
- https://www.khanacademy.org/computing/computer-science/cryptography/modern-crypt/v/intro-to-rsa-encryption
- An Introduction to Modular Math from from Khanacademy: https://www.khanacademy.org/computing/computer-science/cryptography/modarithmetic/a/what-is-modular-arithmetic

 
**CPython Official Source Code Documentation**

Because your text notes that the real Python list is written in C, reading the structural comments left by core Python core developers inside the actual source code provides incredible context.
• File to Look For: listobject.c inside the CPython GitHub repository.
• The Famous Allocation Formula Comment: The file contains the actual C math formula for Python's growth pattern (\(0, 4, 8, 16, 24, 32, \dots\)) alongside comments detailing why they choose a growth rate multiplier of roughly 1.125 (specifically new_allocated = (size_t)newsize + (newsize >> 3) + (newsize < 9 ? 3 : 6);). It details the exact CPU cache line and memory bandwidth tradeoffs mentioned in your text block.

💻



In [1]:
import random

# ==========================================
# 1. CORE MATHEMATICAL UTILITIES
# ==========================================

def gcd(a, b):
    """
    Computes the Greatest Common Divisor (GCD) using Euclid's method.
    Used during key generation to ensure 'e' and (p-1)*(q-1) are relatively prime.
    """
    if b == 0:
        return a
    return gcd(b, a % b)

def ext_gcd(x, y):
    """
    Extended Euclidean Algorithm.
    Returns a tuple (d, a, b) such that d = gcd(x, y) and d = a*x + b*y.
    Used to compute the modular multiplicative inverse for the private key 'd'.
    """
    if y == 0:
        return (x, 1, 0)
    else:
        (d, a, b) = ext_gcd(y, x % y)
        return (d, b, a - (x // y) * b)

def modexp(x, n, p):
    """
    Computes (x^n) % p efficiently using recursive divide-and-conquer.
    Keeps numerical limits small at every step to prevent massive slowdowns.
    """
    if n == 0:
        return 1
    t = (x * x) % p
    tmp = modexp(t, n // 2, p)
    if n % 2 != 0:
        tmp = (tmp * x) % p
    return tmp

# ==========================================
# 2. STRING TO INT CHUNKING LOGIC
# ==========================================

def str_to_chunks(msg, chunk_size):
    """
    Breaks a text string into a list of integers.
    1. Converts string to raw bytes, then to a unified hex string.
    2. Slices the hex string into intervals determined by 'chunk_size' (in hex characters).
    3. Converts each hex block safely into a standard decimal integer.
    """
    # Ensure every single byte maps to exactly 2 hex characters via '02x'
    msg_bytes = bytes(msg, "utf-8")
    hex_str = "".join([f"{b:02x}" for b in msg_bytes])
    
    chunk_list = []
    # Step through the hex string matching your text's index structure
    for i in range(0, len(hex_str), chunk_size):
        chunk_hex = hex_str[i : i + chunk_size]
        if chunk_hex:
            chunk_list.append(chunk_hex)
            
    # Safely convert hex to integer without using eval()
    return [int(x, 16) for x in chunk_list]

def chunks_to_str(chunk_list, chunk_size):
    """
    Reconstructs the original text string from decrypted integer chunks.
    Pads chunks with leading zeros if numerical shifts dropped their length.
    """
    hex_list = []
    for chunk in chunk_list:
        # Strip the standard '0x' prefix from Python's hex utility
        hex_str = hex(chunk)[2:]
        
        # Calculate exactly how many leading zeros are missing.
        # Since 1 byte = 2 hex characters, every chunk needs to be padded
        # up to the expected hex character limit.
        missing_zeros = chunk_size - len(hex_str)
        if missing_zeros > 0:
            hex_str = "0" * missing_zeros + hex_str
            
    # If the total hex string is odd, fix alignment by adding a top-level zero
    hstring = "".join(hex_list) if hex_list else ""
    # Alternative robust concatenation loop to dynamically build cleanly
    hex_string_built = "".join([
        hex(c)[2:].zfill(chunk_size) if i < len(chunk_list)-1 else hex(c)[2:].zfill(len(hex(c)[2:]) + len(hex(c)[2:])%2)
        for i, c in enumerate(chunk_list)
    ])
    
    # Clean fallback sequence to prevent bytearray parity mismatch errors:
    final_hex = []
    for i, chunk in enumerate(chunk_list):
        h = hex(chunk)[2:]
        # The final chunk might naturally be shorter if the string wasn't a perfect multiple
        expected_len = chunk_size if i < len(chunk_list) - 1 else (len(h) + len(h) % 2)
        final_hex.append(h.zfill(expected_len))
        
    msg_array = bytearray.fromhex("".join(final_hex))
    return msg_array.decode("utf-8")

# ==========================================
# 3. RSA INTERFACE FUNCTIONS
# ==========================================

def gen_keys(p, q):
    """
    Generates Public and Private keys given two primes p and q.
    Returns: (e, d, n) where (e, n) is public and (d, n) is private.
    """
    n = p * q
    phi = (p - 1) * (q - 1)
    
    # Pick a random candidate for e
    e = random.randint(2, phi - 1)
    while gcd(phi, e) != 1:
        e = random.randint(2, phi - 1)
        
    # Get the private multiplicative inverse key
    _, _, b = ext_gcd(phi, e)
    if b < 0:
        d = phi + b
    else:
        d = b
    return (e, d, n)

def encrypt(msg, e, n):
    """
    Encrypts a plaintext string using the public key parameters (e, n).
    Dynamically sizes chunks to ensure m < n constraint is kept safe.
    """
    # bit_length // 8 gives us the maximum safe byte allocation size
    byte_size = n.bit_length() // 8
    # Since 1 byte = 2 hex characters, chunk_size in hex characters is byte_size * 2
    hex_chunk_size = byte_size * 2
    
    all_chunks = str_to_chunks(msg, hex_chunk_size)
    return [modexp(msg_chunk, e, n) for msg_chunk in all_chunks]

def decrypt(cipher_chunks, d, n):
    """
    Decrypts a list of cipher integers using private key parameters (d, n).
    """
    byte_size = n.bit_length() // 8
    hex_chunk_size = byte_size * 2
    
    plain_chunks = [modexp(cipher_chunk, d, n) for cipher_chunk in cipher_chunks]
    return chunks_to_str(plain_chunks, hex_chunk_size)

# ==========================================
# 4. TESTING REVISION WORKFLOW
# ==========================================
if __name__ == "__main__":
    # Test case from textbook using p=5563, q=8191
    p_prime = 5563
    q_prime = 8191
    
    print("--- RSA Encryption Initialization ---")
    e, d, n = gen_keys(p_prime, q_prime)
    print(f"Generated Public Exponent (e): {e}")
    print(f"Generated Private Exponent (d): {d}")
    # n will be 45566533 (exactly matching text)
    print(f"Generated Modulus (n): {n}\n")
    
    secret_message = "Python"
    print(f"Original Text: '{secret_message}'")
    
    # Encrypt the string
    encrypted_chunks = encrypt(secret_message, e, n)
    print(f"Encrypted Integer Chunks: {encrypted_chunks}")
    
    # Decrypt back to clear text
    decrypted_message = decrypt(encrypted_chunks, d, n)
    print(f"Decrypted Result: '{decrypted_message}'")


--- RSA Encryption Initialization ---
Generated Public Exponent (e): 38017981
Generated Private Exponent (d): 25962301
Generated Modulus (n): 45566533

Original Text: 'Python'
Encrypted Integer Chunks: [16053328, 6844270]
Decrypted Result: 'Python'


In [2]:
import random

# ==========================================
# 1. LARGE PRIME GENERATION UTILITIES
# ==========================================

def is_prime_miller_rabin(n, k=40):
    """
    Tests if a number is prime using the Miller-Rabin probabilistic test.
    k=40 rounds gives an error probability of less than 4^(-40), which is mathematically negligible.
    """
    if n == 2 or n == 3:
        return True
    if n <= 1 or n % 2 == 0:
        return False

    # Find r and s such that n - 1 = 2^s * r
    s = 0
    r = n - 1
    while r % 2 == 0:
        r //= 2
        s += 1

    # Witness loop
    for _ in range(k):
        a = random.randint(2, n - 2)
        x = modexp(a, r, n)
        if x == 1 or x == n - 1:
            continue
        
        for _ in range(s - 1):
            x = modexp(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def generate_large_prime(bits=1024):
    """
    Generates a secure cryptographically large random prime number of the specified bit length.
    """
    while True:
        # Generate a random integer with the target number of bits
        # Enforce that the highest bit is 1 (to guarantee bit size) and the lowest bit is 1 (odd)
        candidate = random.getrandbits(bits)
        candidate |= (1 << (bits - 1)) | 1
        
        # Fast filter: Check divisibility by small primes before doing heavy math
        small_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]
        if any(candidate % p == 0 for p in small_primes if candidate != p):
            continue
            
        if is_prime_miller_rabin(candidate):
            return candidate

# ==========================================
# 2. CORE MATHEMATICAL UTILITIES
# ==========================================

def gcd(a, b):
    """Computes the Greatest Common Divisor (GCD) using Euclid's method."""
    if b == 0:
        return a
    return gcd(b, a % b)

def ext_gcd(x, y):
    """Extended Euclidean Algorithm returning (d, a, b) where d = a*x + b*y."""
    if y == 0:
        return (x, 1, 0)
    else:
        (d, a, b) = ext_gcd(y, x % y)
        return (d, b, a - (x // y) * b)

def modexp(x, n, p):
    """Computes (x^n) % p efficiently using recursive divide-and-conquer."""
    if n == 0:
        return 1
    t = (x * x) % p
    tmp = modexp(t, n // 2, p)
    if n % 2 != 0:
        tmp = (tmp * x) % p
    return tmp

# ==========================================
# 3. STRING TO INT CHUNKING LOGIC
# ==========================================

def str_to_chunks(msg, chunk_size):
    """Breaks a text string into a list of integers based on hex character chunk sizes."""
    msg_bytes = bytes(msg, "utf-8")
    hex_str = "".join([f"{b:02x}" for b in msg_bytes])
    
    chunk_list = []
    for i in range(0, len(hex_str), chunk_size):
        chunk_hex = hex_str[i : i + chunk_size]
        if chunk_hex:
            chunk_list.append(chunk_hex)
            
    return [int(x, 16) for x in chunk_list]

def chunks_to_str(chunk_list, chunk_size):
    """Reconstructs the original text string from decrypted integer chunks."""
    final_hex = []
    for i, chunk in enumerate(chunk_list):
        h = hex(chunk)[2:]
        # The final chunk might naturally be shorter if the string wasn't a perfect multiple
        expected_len = chunk_size if i < len(chunk_list) - 1 else (len(h) + len(h) % 2)
        final_hex.append(h.zfill(expected_len))
        
    msg_array = bytearray.fromhex("".join(final_hex))
    return msg_array.decode("utf-8")

# ==========================================
# 4. RSA INTERFACE FUNCTIONS
# ==========================================

def gen_keys(p, q):
    """Generates Public and Private keys given two primes p and q."""
    n = p * q
    phi = (p - 1) * (q - 1)
    
    # Standard choice for e is 65537 because it makes encryption fast. 
    # If it happens to share a factor with phi, we fall back to a random search.
    e = 65537
    if gcd(phi, e) != 1:
        e = random.randint(2, phi - 1)
        while gcd(phi, e) != 1:
            e = random.randint(2, phi - 1)
        
    _, _, b = ext_gcd(phi, e)
    d = phi + b if b < 0 else b
    return (e, d, n)

def encrypt(msg, e, n):
    """Encrypts a plaintext string using public key parameters (e, n)."""
    byte_size = n.bit_length() // 8
    hex_chunk_size = byte_size * 2
    
    all_chunks = str_to_chunks(msg, hex_chunk_size)
    return [modexp(msg_chunk, e, n) for msg_chunk in all_chunks]

def decrypt(cipher_chunks, d, n):
    """Decrypts a list of cipher integers using private key parameters (d, n)."""
    byte_size = n.bit_length() // 8
    hex_chunk_size = byte_size * 2
    
    plain_chunks = [modexp(cipher_chunk, d, n) for cipher_chunk in cipher_chunks]
    return chunks_to_str(plain_chunks, hex_chunk_size)

# ==========================================
# 5. EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    print("--- 1. Generating 1024-bit Primes (This may take a moment) ---")
    p = generate_large_prime(bits=1024)
    q = generate_large_prime(bits=1024)
    print(f"Prime p generated ({p.bit_length()} bits)")
    print(f"Prime q generated ({q.bit_length()} bits)")
    
    print("\n--- 2. Building Key pairs ---")
    e, d, n = gen_keys(p, q)
    print(f"Modulus n size: {n.bit_length()} bits")
    print(f"Public Exponent e: {e}")
    
    # A single chunk can now hold a very long paragraph since n is massive!
    secret_message = "This is a highly secure message encrypted using an automated 1024-bit RSA key pair generator."
    print(f"\nOriginal Text:\n'{secret_message}'")
    
    print("\n--- 3. Running Encryption ---")
    encrypted_chunks = encrypt(secret_message, e, n)
    print(f"Ciphertext Array (Number of chunks: {len(encrypted_chunks)}):")
    print(encrypted_chunks)
    
    print("\n--- 4. Running Decryption ---")
    decrypted_message = decrypt(encrypted_chunks, d, n)
    print(f"Decrypted Result:\n'{decrypted_message}'")


--- 1. Generating 1024-bit Primes (This may take a moment) ---
Prime p generated (1024 bits)
Prime q generated (1024 bits)

--- 2. Building Key pairs ---
Modulus n size: 2047 bits
Public Exponent e: 65537

Original Text:
'This is a highly secure message encrypted using an automated 1024-bit RSA key pair generator.'

--- 3. Running Encryption ---
Ciphertext Array (Number of chunks: 1):
[13054825323490841666183096696124683199483294863772259072942140712171111766656620994194301597119729994090618519123043626610184315265896530405809005329560073135202281069345859713361568045147092951183516846230007805048987069592420873414668667782904589008392985120525789659245827504344412800630239849499582943105831599636716634237555321558029398088032784996563652966503811818141054911813609741701893413270428307207397988520935580811207912175663021763298271358248132578302300309206634112612236526023093813539663932851918592429594416668787388700056218679970827414847735159424083678316292873494017565678206613728650797